# PartA_1

This notebook implements the baseline CNN, augmentation variants, and Random Erasing model for the flower dataset.

Structure:
1. Utilities and data loading
2. Model definitions
3. Training benchmark and evaluation

## Colab Setup

In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")
# !unzip "/content/gdrive/MyDrive/Deep_learning_2/flower.h5.zip" -d "/content/gdrive/MyDrive/Deep_learning_2/"
!ls /content/gdrive/MyDrive/Deep_learning_2


## Utilities

This section contains the imports, the HDF5 loader, plotting utilities, and helper functions reused across all other notebook.

In [ ]:
import csv
import time
import h5py
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
from keras import layers
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.models import Model

# source: https://www.tensorflow.org/guide/keras/functional_api
# source: https://www.tensorflow.org/tutorials/images/data_augmentation
# source: https://www.tensorflow.org/api_docs/python/tf/keras/layers/RandomFlip
# source: https://www.tensorflow.org/api_docs/python/tf/keras/layers/RandomZoom
# source: https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/ModelCheckpoint
# source: https://keras.io/api/layers/preprocessing_layers/image_augmentation/random_erasing/
# source: https://www.tensorflow.org/api_docs/python/tfm/vision/augment/MixupAndCutmix


In [ ]:
def loadDataH5():
    candidate_paths = [
        "data1.h5",
        "/content/data1.h5",
        "/content/gdrive/MyDrive/Deep_learning_2/data1.h5",
        "/content/drive/MyDrive/Deep_learning_2/data1.h5",
    ]

    data_path = None
    for candidate in candidate_paths:
        if os.path.exists(candidate):
            data_path = candidate
            break

    if data_path is None:
        raise FileNotFoundError(
            "Could not find data1.h5. Expected it in the current directory, "
            "/content, or /content/gdrive/MyDrive/Deep_learning_2."
        )

    print("Using data file:", data_path)

    with h5py.File(data_path, "r") as hf:
        trainX = np.array(hf.get("trainX"))
        trainY = np.array(hf.get("trainY"))
        valX = np.array(hf.get("valX"))
        valY = np.array(hf.get("valY"))

    print("trainX shape:", trainX.shape, "trainY shape:", trainY.shape)
    print("valX shape:", valX.shape, "valY shape:", valY.shape)
    return trainX, trainY, valX, valY


def plot_history(history, model_name, output_dir="plots"):
    os.makedirs(output_dir, exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].plot(history["accuracy"], label="train_accuracy")
    axes[0].plot(history["val_accuracy"], label="val_accuracy")
    axes[0].set_title(f"{model_name} Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()

    axes[1].plot(history["loss"], label="train_loss")
    axes[1].plot(history["val_loss"], label="val_loss")
    axes[1].set_title(f"{model_name} Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{model_name}_history.png"), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def plot_confusion_matrix(y_true, y_pred, model_name, class_names, output_dir="plots"):
    os.makedirs(output_dir, exist_ok=True)
    cm = confusion_matrix(y_true, y_pred)

    row_sums = cm.sum(axis=1, keepdims=True)
    cm_normalized = np.divide(
        cm.astype("float"),
        row_sums,
        out=np.zeros_like(cm, dtype=float),
        where=row_sums != 0,
    )

    fig, ax = plt.subplots(figsize=(10, 8))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=True)
    ax.set_title(f"{model_name} Confusion Matrix")
    plt.tight_layout()
    plt.savefig(
        os.path.join(output_dir, f"{model_name}_confusion_matrix.png"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(10, 8))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm_normalized,
        display_labels=class_names,
    )
    disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=True, values_format=".2f")
    ax.set_title(f"{model_name} Normalized Confusion Matrix")
    plt.tight_layout()
    plt.savefig(
        os.path.join(output_dir, f"{model_name}_confusion_matrix_normalized.png"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

    np.savetxt(
        os.path.join(output_dir, f"{model_name}_confusion_matrix.csv"),
        cm,
        delimiter=",",
        fmt="%d",
    )
    np.savetxt(
        os.path.join(output_dir, f"{model_name}_confusion_matrix_normalized.csv"),
        cm_normalized,
        delimiter=",",
        fmt="%.6f",
    )

    return cm, cm_normalized


def save_classification_report(y_true, y_pred, class_names, model_name, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    report_text = classification_report(y_true, y_pred, target_names=class_names)
    report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)

    txt_path = os.path.join(output_dir, f"{model_name}_classification_report.txt")
    with open(txt_path, "w") as report_file:
        report_file.write(report_text)

    csv_path = os.path.join(output_dir, f"{model_name}_classification_report.csv")
    with open(csv_path, "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["label", "precision", "recall", "f1-score", "support"])
        for label, metrics in report_dict.items():
            if isinstance(metrics, dict):
                writer.writerow([
                    label,
                    metrics.get("precision"),
                    metrics.get("recall"),
                    metrics.get("f1-score"),
                    metrics.get("support"),
                ])


def plot_prediction_examples(
    images,
    y_true,
    y_pred,
    true_class,
    pred_class=None,
    max_images=6,
    model_name="model",
    output_dir="plots",
):
    os.makedirs(output_dir, exist_ok=True)

    if pred_class is None:
        indices = np.where((y_true == true_class) & (y_pred == true_class))[0]
        title = f"{model_name}: correctly classified class {true_class}"
        file_name = f"{model_name}_class_{true_class}_correct_examples.png"
    else:
        indices = np.where((y_true == true_class) & (y_pred == pred_class))[0]
        title = f"{model_name}: class {true_class} misclassified as class {pred_class}"
        file_name = f"{model_name}_class_{true_class}_pred_{pred_class}_examples.png"

    if len(indices) == 0:
        print(f"No matching examples found for {title}.")
        return

    indices = indices[:max_images]
    fig, axes = plt.subplots(1, len(indices), figsize=(3 * len(indices), 3))
    if len(indices) == 1:
        axes = [axes]

    for ax, idx in zip(axes, indices):
        ax.imshow(images[idx])
        ax.set_title(f"true={y_true[idx]}\npred={y_pred[idx]}")
        ax.axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, file_name), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def get_top_confusion_pair(cm):
    confusion_only = cm.copy()
    np.fill_diagonal(confusion_only, 0)
    max_index = np.argmax(confusion_only)
    true_class, pred_class = np.unravel_index(max_index, confusion_only.shape)
    if confusion_only[true_class, pred_class] == 0:
        return None
    return true_class, pred_class


def plot_class_confusion_comparison(
    images,
    y_true,
    y_pred,
    true_class,
    pred_class,
    max_images=4,
    model_name="model",
    output_dir="plots",
):
    os.makedirs(output_dir, exist_ok=True)

    misclassified_idx = np.where((y_true == true_class) & (y_pred == pred_class))[0]
    true_correct_idx = np.where((y_true == true_class) & (y_pred == true_class))[0]
    pred_correct_idx = np.where((y_true == pred_class) & (y_pred == pred_class))[0]

    if len(misclassified_idx) == 0:
        print(
            f"No misclassified examples found for class {true_class} predicted as class {pred_class}."
        )
        return

    misclassified_idx = misclassified_idx[:max_images]
    true_correct_idx = true_correct_idx[:max_images]
    pred_correct_idx = pred_correct_idx[:max_images]

    columns = max(len(misclassified_idx), len(true_correct_idx), len(pred_correct_idx), 1)
    fig, axes = plt.subplots(3, columns, figsize=(3 * columns, 9))

    if columns == 1:
        axes = np.array(axes).reshape(3, 1)

    row_titles = [
        f"Misclassified: true={true_class}, pred={pred_class}",
        f"Correct examples of true class {true_class}",
        f"Correct examples of predicted class {pred_class}",
    ]
    row_indices = [misclassified_idx, true_correct_idx, pred_correct_idx]

    for row, (title, indices) in enumerate(zip(row_titles, row_indices)):
        for col in range(columns):
            ax = axes[row, col]
            if col < len(indices):
                idx = indices[col]
                ax.imshow(images[idx])
                ax.set_title(f"true={y_true[idx]}\npred={y_pred[idx]}")
                ax.axis("off")
            else:
                ax.axis("off")
        axes[row, 0].set_ylabel(title, rotation=90, fontsize=11, labelpad=20)

    plt.suptitle(
        f"{model_name}: visual comparison for confusion {true_class} -> {pred_class}",
        fontsize=14,
    )
    plt.tight_layout()
    plt.savefig(
        os.path.join(
            output_dir,
            f"{model_name}_class_{true_class}_vs_class_{pred_class}_comparison.png",
        ),
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)


## Models

This section defines the baseline CNN, the geometric augmentation variants, and the Random Erasing variant.

In [ ]:
def cnn_classifier(x, num_classes):
    x = layers.Conv2D(32, (3, 3), padding="same", activation="relu")(x)
    x = layers.MaxPooling2D(pool_size=(3, 3), strides=2)(x)
    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = layers.MaxPooling2D(pool_size=(3, 3), strides=2)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    return layers.Dense(num_classes, activation="softmax")(x)


def basicModel(input_shape=(128, 128, 3), num_classes=17):
    inputs = tf.keras.Input(shape=input_shape, name="input_image")
    outputs = cnn_classifier(inputs, num_classes)
    return Model(inputs=inputs, outputs=outputs, name="basicModel")


flip_augmentation = tf.keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
    ],
    name="flip_augmentation",
)


def augmentedModel(input_shape=(128, 128, 3), num_classes=17):
    inputs = tf.keras.Input(shape=input_shape, name="input_image")
    x = flip_augmentation(inputs)
    outputs = cnn_classifier(x, num_classes)
    return Model(inputs=inputs, outputs=outputs, name="augmentedModel")


zoom_augmentation = tf.keras.Sequential(
    [
        layers.RandomZoom(height_factor=0.1, width_factor=0.1),
    ],
    name="zoom_augmentation",
)


def zoomModel(input_shape=(128, 128, 3), num_classes=17):
    inputs = tf.keras.Input(shape=input_shape, name="input_image")
    x = zoom_augmentation(inputs)
    outputs = cnn_classifier(x, num_classes)
    return Model(inputs=inputs, outputs=outputs, name="zoomModel")


random_erasing_augmentation = tf.keras.Sequential(
    [
        layers.RandomErasing(
            factor=0.5,
            scale=(0.02, 0.15),
            fill_value=0.0,
            value_range=(0, 1),
        ),
    ],
    name="random_erasing_augmentation",
)


def randomErasingModel(input_shape=(128, 128, 3), num_classes=17):
    inputs = tf.keras.Input(shape=input_shape, name="input_image")
    x = random_erasing_augmentation(inputs)
    outputs = cnn_classifier(x, num_classes)
    return Model(inputs=inputs, outputs=outputs, name="randomErasingModel")

combinedGeometricAugmentation = tf.keras.Sequential([
  layers.RandomFlip("horizontal_and_vertical"),
  layers.RandomRotation(0.2),
])

def combinedGeometricAugmentationModel(input_shape=(128, 128, 3), num_classes=17):
    inputs = tf.keras.Input(shape=input_shape, name="input_image")
    x = combinedGeometricAugmentation(inputs)
    outputs = cnn_classifier(x, num_classes)
    return Model(inputs=inputs, outputs=outputs, name="combinedGeometricAugmentationModel")

## Benchmark

This section trains all Part A.1 variants, evaluates them on the validation set, and produces plots, confusion matrices, and visual comparison figures.

In [ ]:
def train_and_evaluate_models(trainX, trainY, valX, valY, epochs=20, batch_size=32):
    models_to_train = {
        "basicModel": basicModel,
        "augmentedModel": augmentedModel,
        "zoomModel": zoomModel,
        "randomErasingModel": randomErasingModel,
        "combinedGeometricAugmentationModel": combinedGeometricAugmentationModel,
    }

    histories = {}
    results = {}
    class_names = [f"Class {index}" for index in range(17)]

    for model_name, model_builder in models_to_train.items():
        print(f"\n{'=' * 60}")
        print(f"Training {model_name}")
        print(f"{'=' * 60}")

        model = model_builder(num_classes=17)
        model.summary()

        model.compile(
            optimizer="adam",
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"],
        )

        checkpoint_cb = ModelCheckpoint(
            filepath=f"{model_name}_best.keras",
            monitor="val_accuracy",
            save_best_only=True,
            mode="max",
            verbose=1,
        )

        train_start = time.perf_counter()
        history = model.fit(
            trainX,
            trainY,
            validation_data=(valX, valY),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=[checkpoint_cb],
        )
        training_seconds = time.perf_counter() - train_start

        best_model = tf.keras.models.load_model(f"{model_name}_best.keras")
        eval_start = time.perf_counter()
        val_loss, val_accuracy = best_model.evaluate(valX, valY, verbose=0)
        predict_start = time.perf_counter()
        y_pred_probs = best_model.predict(valX, verbose=0)
        predict_seconds = time.perf_counter() - predict_start
        evaluation_seconds = time.perf_counter() - eval_start
        y_pred = np.argmax(y_pred_probs, axis=1)

        histories[model_name] = history.history
        results[model_name] = {
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
            "training_seconds": training_seconds,
            "evaluation_seconds": evaluation_seconds,
            "predict_seconds": predict_seconds,
            "predict_ms_per_image": (predict_seconds / len(valX)) * 1000,
        }

        print(f"\n{model_name} validation loss: {val_loss:.4f}")
        print(f"{model_name} validation accuracy: {val_accuracy:.4f}")
        print(f"{model_name} training time: {training_seconds:.2f}s")
        print(f"{model_name} prediction time: {predict_seconds:.4f}s ({(predict_seconds / len(valX)) * 1000:.3f} ms/image)")

        print(f"\nClassification report for {model_name}:")
        print(classification_report(valY, y_pred, target_names=class_names))
        save_classification_report(valY, y_pred, class_names, model_name, output_dir="plots")

        plot_history(history.history, model_name)
        cm, cm_normalized = plot_confusion_matrix(valY, y_pred, model_name, class_names)

        best_class = int(np.argmax(np.diag(cm_normalized)))
        best_class_score = np.diag(cm_normalized)[best_class]
        print(
            f"Best classified class for {model_name}: "
            f"class {best_class} with normalized recall {best_class_score:.2f}"
        )
        plot_prediction_examples(
            valX,
            valY,
            y_pred,
            true_class=best_class,
            pred_class=None,
            max_images=5,
            model_name=model_name,
        )

        top_confusion = get_top_confusion_pair(cm)
        if top_confusion is not None:
            true_class, pred_class = top_confusion
            plot_prediction_examples(
                valX,
                valY,
                y_pred,
                true_class=true_class,
                pred_class=pred_class,
                max_images=5,
                model_name=model_name,
            )
            plot_class_confusion_comparison(
                valX,
                valY,
                y_pred,
                true_class=true_class,
                pred_class=pred_class,
                max_images=4,
                model_name=model_name,
            )

    print("\nFinal results summary:")
    for model_name, metrics in results.items():
        print(
            f"{model_name}: val_loss={metrics['val_loss']:.4f}, "
            f"val_accuracy={metrics['val_accuracy']:.4f}, "
            f"train_time={metrics['training_seconds']:.2f}s, "
            f"predict_time={metrics['predict_seconds']:.4f}s"
        )

    return histories, results


## Run the Benchmark

Execute the following cell to load the data and train all Part A.1 models.

In [ ]:
trainX, trainY, valX, valY = loadDataH5()
histories, results = train_and_evaluate_models(
    trainX,
    trainY,
    valX,
    valY,
    epochs=20,
    batch_size=32,
)

## Export Results to CSV

Export the main metrics to CSV after running the benchmark.

In [ ]:
import csv
import os
import shutil
import time


def export_parta1_results(histories, results, output_dir="exports_partA1", copy_to_drive=True):
    os.makedirs(output_dir, exist_ok=True)

    summary_path = os.path.join(output_dir, "partA1_summary.csv")
    with open(summary_path, "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["model_name", "val_loss", "val_accuracy", "training_seconds", "evaluation_seconds", "predict_seconds", "predict_ms_per_image"])
        for model_name, metrics in results.items():
            writer.writerow([
                model_name,
                metrics.get("val_loss"),
                metrics.get("val_accuracy"),
                metrics.get("training_seconds"),
                metrics.get("evaluation_seconds"),
                metrics.get("predict_seconds"),
                metrics.get("predict_ms_per_image"),
            ])

    history_path = os.path.join(output_dir, "partA1_history.csv")
    with open(history_path, "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["model_name", "epoch", "accuracy", "val_accuracy", "loss", "val_loss"])
        for model_name, history in histories.items():
            epochs = len(history.get("accuracy", []))
            for epoch in range(epochs):
                writer.writerow([
                    model_name,
                    epoch + 1,
                    history.get("accuracy", [None] * epochs)[epoch],
                    history.get("val_accuracy", [None] * epochs)[epoch],
                    history.get("loss", [None] * epochs)[epoch],
                    history.get("val_loss", [None] * epochs)[epoch],
                ])

    master_path = os.path.join(output_dir, "partA1_master_summary.csv")
    with open(master_path, "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["question", "model_name", "val_loss", "val_accuracy", "training_seconds", "predict_seconds", "predict_ms_per_image", "short_comment"])
        for model_name, metrics in results.items():
            writer.writerow(["PartA_1", model_name, metrics.get("val_loss"), metrics.get("val_accuracy"), metrics.get("training_seconds"), metrics.get("predict_seconds"), metrics.get("predict_ms_per_image"), ""])

    print(f"Saved CSV files to {output_dir}")
    print(summary_path)
    print(history_path)

    if copy_to_drive:
        drive_targets = [
            "/content/gdrive/MyDrive/Deep_learning_2",
            "/content/drive/MyDrive/Deep_learning_2",
        ]
        copied = False
        for drive_dir in drive_targets:
            if os.path.isdir(drive_dir):
                drive_export_dir = os.path.join(drive_dir, output_dir)
                os.makedirs(drive_export_dir, exist_ok=True)
                shutil.copy2(summary_path, os.path.join(drive_export_dir, os.path.basename(summary_path)))
                shutil.copy2(history_path, os.path.join(drive_export_dir, os.path.basename(history_path)))
                print(f"Copied CSV files to {drive_export_dir}")
                plot_source_dir = "plots"
                for checkpoint_name in ["basicModel_best.keras", "augmentedModel_best.keras", "zoomModel_best.keras", "randomErasingModel_best.keras"]:
                    if os.path.isfile(checkpoint_name):
                        shutil.copy2(checkpoint_name, os.path.join(drive_dir, checkpoint_name))
                if os.path.isdir(plot_source_dir):
                    drive_plot_dir = os.path.join(drive_dir, plot_source_dir)
                    os.makedirs(drive_plot_dir, exist_ok=True)
                    for file_name in os.listdir(plot_source_dir):
                        source_file = os.path.join(plot_source_dir, file_name)
                        if os.path.isfile(source_file):
                            shutil.copy2(source_file, os.path.join(drive_plot_dir, file_name))
                    print(f"Copied plot files to {drive_plot_dir}")
                copied = True
                break
        if not copied:
            print("Google Drive export folder not found. CSV files were saved locally only.")


if "histories" in globals() and "results" in globals():
    export_parta1_results(histories, results)
else:
    print("Run the benchmark cell first, then rerun this export cell.")
